# Dedup (corpus) — transparency pipeline eval notebook

Tests the `/dedup/corpus` endpoint (whole-collection clustering) used in **Phase B** of the
transparency pipeline. After all documents are ingested, the orchestrator calls this endpoint
twice — once for entity clusters, once for relation clusters — before merging and re-describing
each cluster with `summarizer[aggregate]`.

**Algorithm**: union-find over near-neighbour edges.

| Band | Condition | Action |
|------|-----------|--------|
| Auto-merge | edge score ≥ `CORPUS_MERGE` (0.92) | union unconditionally |
| LLM band   | `CORPUS_LLM_LOW` (0.80) ≤ score < 0.92 | LLM adjudication — `yes` → union |
| No-merge   | score < `CORPUS_LLM_LOW` | skip |

Returns `clusters: list[list[id]]`. Every object appears in exactly one cluster
(singletons are 1-element clusters).

**Edge cases covered**: empty collection, entity auto-merge pairs, transitive entity chain,
isolated singleton, relation LLM band, `type_filter` isolation, mixed collection (no cross-type
merges expected).

**Domain**: financial transparency pipeline (`financial_flows` use case — PERSON and
ORGANIZATION entities, PAYMENT_TO relations).

> Advisory cases (LLM band) require Ollama to be running.  
> `WEAVIATE_URL` must resolve to the same Weaviate instance the NLP service uses internally.

In [ ]:
import os, sys, logging
from itertools import combinations
import requests

# --- endpoints -----------------------------------------------------------
NLP_BASE_URL   = os.environ.get('NLP_BASE_URL', 'https://wiig.dia.fi.upm.es/nlp').rstrip('/')
WEAVIATE_URL   = os.environ.get('WEAVIATE_URL', 'http://localhost:8080').rstrip('/')
VECTORIZER_URL = os.environ.get('VECTORIZER_URL', f'{NLP_BASE_URL}/vectorizer/embed')
ENDPOINT       = f'{NLP_BASE_URL}/dedup/corpus'
HEADERS        = {'Content-Type': 'application/json',
                  'X-API-Key': os.environ.get('NLP_API_KEY', '')}
TIMEOUT        = 120   # corpus clustering can take longer

# --- service thresholds (mirror nlp/dedup/service.py defaults) ----------
CORPUS_MERGE   = float(os.environ.get('DEDUP_CORPUS_MERGE',   '0.92'))
CORPUS_LLM_LOW = float(os.environ.get('DEDUP_CORPUS_LLM_LOW', '0.80'))
TOP_K          = int(os.environ.get('DEDUP_TOP_K', '10'))

COLLECTION       = 'DedupCorpusEval'
EMPTY_COLLECTION = 'DedupCorpusEvalEmpty'

# --- logging -------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format='%(name)-24s %(levelname)-8s  %(message)s',
    stream=sys.stdout, force=True,
)
for _noisy in ('httpx', 'httpcore', 'urllib3', 'requests', 'charset_normalizer'):
    logging.getLogger(_noisy).setLevel(logging.WARNING)
log = logging.getLogger('eval.dedup.corpus')

print(f'NLP        → {ENDPOINT}')
print(f'VECTORIZER → {VECTORIZER_URL}')
print(f'WAV        → {WEAVIATE_URL}  (must be the same instance the NLP service sees)')
print(f'thresholds:  merge ≥ {CORPUS_MERGE}   llm_low ≥ {CORPUS_LLM_LOW}   top_k = {TOP_K}')

In [ ]:
# ── SETUP: create Weaviate collections and insert test corpus ─────────────

# --- Weaviate v1 REST helpers -------------------------------------------
def _wv(path, method='GET', body=None):
    url = f'{WEAVIATE_URL}/v1/{path}'
    fn  = {'GET': requests.get, 'POST': requests.post, 'DELETE': requests.delete}[method]
    kw  = {'timeout': 15}
    if body is not None:
        kw['json'] = body
    r = fn(url, **kw)
    r.raise_for_status()
    return r.json() if r.content else {}


def _ensure_collection(name, drop_existing=False):
    schema = {
        'class': name, 'vectorizer': 'none',
        'properties': [
            {'name': 'canonical_name', 'dataType': ['text']},
            {'name': 'description',    'dataType': ['text']},
            {'name': 'type',           'dataType': ['text']},
        ],
    }
    try:
        _wv('schema', 'POST', schema)
        log.info('created collection  %s', name)
    except requests.HTTPError as e:
        if e.response.status_code == 422 and drop_existing:
            log.warning('%s already exists — dropping and re-creating', name)
            _wv(f'schema/{name}', 'DELETE')
            _wv('schema', 'POST', schema)
        else:
            raise


def _insert(collection, props, vector):
    r = _wv('objects', 'POST',
            {'class': collection, 'properties': props, 'vector': vector})
    return r['id']


# --- Vectorizer ----------------------------------------------------------
def embed(text):
    """Call /vectorizer/embed and return the embedding as a plain list."""
    r = requests.post(VECTORIZER_URL, json={'text': text}, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()['embedding']


# --- Corpus: 6 entities + 3 relations (financial_flows domain) ----------
#
# Entity pairs/chains designed for controlled clustering:
#   e1, e2  → same PERSON (near-identical descriptions) → expected auto-merge
#   e3      → different PERSON, isolated → expected singleton
#   e4, e5, e6 → same ORGANIZATION with slight description variants → expected transitive chain
#
# Relation pairs designed for the LLM band:
#   r1, r2  → same PAYMENT_TO event, differently worded → advisory (LLM band)
#   r3      → different relation → expected singleton

_CORPUS = [
    # --- Entities (type=entity) ----------------------------------------
    ('e1', 'entity', 'Juan García López',
     'Juan García López, concejal del Ayuntamiento de Madrid desde 2019, '
     'implicado en la adjudicación de contratos de infraestructura urbana '
     'a empresas privadas del sector de la construcción.'),

    ('e2', 'entity', 'Juan García López',
     'Juan García López, edil del Ayuntamiento de Madrid desde 2019, '
     'señalado por adjudicaciones de contratos de obras urbanas '
     'a empresas constructoras privadas.'),        # same person, near-identical wording → auto-merge

    ('e3', 'entity', 'Ana Martínez Ruiz',
     'Ana Martínez Ruiz, presidenta de la Cámara de Contratistas de la Comunidad de Madrid, '
     'experta en regulación de licitaciones públicas y contratación administrativa.'),   # isolated

    ('e4', 'entity', 'Constructora del Sur S.L.',
     'Constructora del Sur S.L., empresa constructora radicada en Sevilla, '
     'adjudicataria de contratos públicos de infraestructura viaria desde 2018.'),

    ('e5', 'entity', 'Constructora del Sur S.L.',
     'Constructora del Sur, Sociedad Limitada, empresa de obras públicas con sede en Sevilla, '
     'beneficiaria de contratos de infraestructura viaria adjudicados desde 2018.'),  # auto-merge with e4

    ('e6', 'entity', 'Constructora del Sur S.L.',
     'Constructora del Sur S.L., constructora sevillana especializada en infraestructura '
     'viaria, receptora de contratos públicos de obras desde 2018.'),  # auto-merge with e4, e5

    # --- Relations (type=relation) -------------------------------------
    ('r1', 'relation', 'PAYMENT_TO: García López → Constructora del Sur',
     'Pago de 450.000 euros efectuado por Juan García López a Constructora del Sur S.L. '
     'en concepto de honorarios por contrato de infraestructura vial adjudicado en 2022.'),

    ('r2', 'relation', 'PAYMENT_TO: García López → Constructora del Sur',
     'Transferencia de 450.000 euros realizada por Juan García López '
     'a Constructora del Sur S.L., vinculada a un contrato de infraestructura '
     'vial del año 2022.'),    # same event, different wording → advisory (LLM band)

    ('r3', 'relation', 'CONTRACT: Martínez → servicios urbanísticos',
     'Contrato de asesoría por 80.000 euros suscrito entre Ana Martínez Ruiz '
     'y una empresa de servicios urbanísticos no identificada en 2023.'),  # isolated
]

# Connectivity sanity check
try:
    _wv('.well-known/ready')
    log.info('Weaviate reachable')
except Exception as exc:
    log.error('Weaviate not reachable at %s  →  %s', WEAVIATE_URL, exc)
    raise

# Create collections
_ensure_collection(COLLECTION,       drop_existing=True)
_ensure_collection(EMPTY_COLLECTION, drop_existing=True)   # stays empty (edge case)

# Embed and insert corpus; store IDs for result interpretation
IDS  = {}   # key → weaviate object id

for _key, _type, _name, _desc in _CORPUS:
    vec = embed(_desc)
    oid = _insert(COLLECTION,
                  {'canonical_name': _name, 'description': _desc, 'type': _type},
                  vec)
    IDS[_key] = oid
    log.info('inserted  %s  type=%-8s  id=%s…  dim=%d', _key, _type, oid[:8], len(vec))

n_entities  = sum(1 for k, t, *_ in _CORPUS if t == 'entity')
n_relations = sum(1 for k, t, *_ in _CORPUS if t == 'relation')
print(f'\n{COLLECTION:<26} {len(_CORPUS)} objects  ({n_entities} entities, {n_relations} relations)')
print(f'{EMPTY_COLLECTION:<26} 0 objects  (empty-corpus edge case)')

In [ ]:
# --- Service call + render + scoring helpers ----------------------------

def call_dedup_corpus(*, collection=COLLECTION, kind='entity',
                      compare_property='description', type_filter=None):
    body = {
        'collection':       collection,
        'kind':             kind,
        'compare_property': compare_property,
    }
    if type_filter is not None:
        body['type_filter'] = type_filter
    r = requests.post(ENDPOINT, json=body, headers=HEADERS, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


# --- ID → key lookup
_ID2KEY = {oid: k for k, oid in IDS.items()}

def _cluster_to_keys(cluster):
    return frozenset(_ID2KEY.get(oid, oid[:8] + '…') for oid in cluster)


def render(resp):
    clusters = resp.get('clusters', [])
    non_singleton = [c for c in clusters if len(c) > 1]
    singletons    = [c for c in clusters if len(c) == 1]

    log.info('clusters total=%d  merged=%d  singletons=%d',
             len(clusters), len(non_singleton), len(singletons))

    print(f'  total clusters : {len(clusters)}')
    print(f'  merged (>1)    : {len(non_singleton)}')
    print(f'  singletons     : {len(singletons)}')

    if non_singleton:
        print('  MERGED CLUSTERS')
        for cl in sorted(non_singleton, key=len, reverse=True):
            keys = sorted(_cluster_to_keys(cl))
            print(f'    {{{"  ".join(keys)}}}  ({len(cl)} items)')

    if singletons:
        print('  SINGLETONS')
        for cl in singletons:
            keys = sorted(_cluster_to_keys(cl))
            print(f'    {{{keys[0]}}}')


def pairwise_metrics(actual_clusters, expected_key_sets):
    """Pairwise precision / recall / F1 for clustering quality."""
    actual_key_clusters = [_cluster_to_keys(cl) for cl in actual_clusters]

    actual_pairs   = set()
    for cl in actual_key_clusters:
        for a, b in combinations(sorted(cl), 2):
            actual_pairs.add((a, b))

    expected_pairs = set()
    for cl in expected_key_sets:
        for a, b in combinations(sorted(cl), 2):
            expected_pairs.add((a, b))

    tp = len(actual_pairs & expected_pairs)
    fp = len(actual_pairs - expected_pairs)
    fn = len(expected_pairs - actual_pairs)

    p  = tp / (tp + fp) if (tp + fp) else 1.0
    r  = tp / (tp + fn) if (tp + fn) else 1.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return tp, fp, fn, p, r, f1


def score_case(resp, expected_key_sets, *, advisory=False):
    """Compare actual clusters vs expected key-sets.

    Returns (passed, tp, fp, fn, precision, recall, f1).
    Advisory cases never fail the scorecard.
    """
    clusters = resp.get('clusters', [])
    actual   = [_cluster_to_keys(cl) for cl in clusters]

    if not expected_key_sets:                        # empty collection: expect zero clusters
        passed = len(clusters) == 0
        return passed, 0, 0, 0, 1.0, 1.0, 1.0

    tp, fp, fn, p, r, f1 = pairwise_metrics(clusters, expected_key_sets)
    passed = (fp == 0 and fn == 0) if not advisory else (fp == 0)  # advisory: FP is hard fail
    return passed, tp, fp, fn, p, r, f1


# --- Cases ---------------------------------------------------------------
# Expected clusters are defined as sets of corpus item keys.
# Singletons must be listed explicitly so the scorer knows which items are expected.
CASES = [
    {
        'id':          'dedup-c-001',
        'description': 'empty collection — fetch_all yields nothing → 0 clusters',
        'params':      {'collection': EMPTY_COLLECTION, 'kind': 'entity'},
        'expected':    [],           # zero clusters expected
    },
    {
        'id':          'dedup-c-002',
        'description': 'entity clustering — auto-merge pair, transitive chain, isolated singleton',
        'params':      {'kind': 'entity', 'type_filter': 'entity'},
        'expected':    [{'e1', 'e2'}, {'e3'}, {'e4', 'e5', 'e6'}],
        'notes':       'e1+e2 same person; e4+e5+e6 same org (transitive); e3 isolated',
    },
    {
        'id':          'dedup-c-003',
        'description': 'relation clustering — same-event pair in LLM band, isolated singleton  [advisory]',
        'params':      {'kind': 'relation', 'type_filter': 'relation'},
        'expected':    [{'r1', 'r2'}, {'r3'}],
        'advisory':    True,
        'notes':       'r1+r2 same payment event (may fall in LLM band); r3 isolated',
    },
    {
        'id':          'dedup-c-004',
        'description': 'no type_filter — entities and relations cluster within their type, no cross-type merges',
        'params':      {'kind': 'entity'},
        'expected':    [{'e1', 'e2'}, {'e3'}, {'e4', 'e5', 'e6'}, {'r1', 'r2'}, {'r3'}],
        'advisory':    True,   # r1+r2 merge is still advisory
        'notes':       'entity and relation descriptions occupy different semantic regions → '
                       'no cross-type edges expected',
    },
]

print(f'{len(CASES)} cases')
for c in CASES:
    adv = '  [advisory]' if c.get('advisory') else ''
    print(f"  {c['id']}: {c['description']}{adv}")

In [ ]:
results = []   # (case_id, passed, advisory, resp_or_None, tp, fp, fn, p, r, f1)

for case in CASES:
    adv_tag = ' [advisory]' if case.get('advisory') else ''
    print(f"── {case['id']}: {case['description']}{adv_tag}")
    try:
        resp = call_dedup_corpus(**case['params'])
    except Exception as exc:
        print(f'   ERROR: {exc}')
        if case.get('notes'):
            print(f'   note: {case["notes"]}')
        print()
        results.append((case['id'], False, case.get('advisory', False),
                        None, 0, 0, 0, 0.0, 0.0, 0.0))
        continue

    render(resp)

    passed, tp, fp, fn, p, r, f1 = score_case(
        resp, case['expected'], advisory=case.get('advisory', False))

    icon  = '✅' if passed else ('⚠️ ' if case.get('advisory') else '❌')
    label = 'pass' if passed else ('advisory mismatch' if case.get('advisory') else 'FAIL')
    print(f'   {icon} {label}  P={p:.2f}  R={r:.2f}  F1={f1:.2f}'
          + (f'  (FP={fp} FN={fn})' if fp or fn else ''))
    if case.get('notes'):
        print(f'   note: {case["notes"]}')
    print()
    results.append((case['id'], passed, case.get('advisory', False),
                    resp, tp, fp, fn, p, r, f1))

In [ ]:
# ── SCORECARD + METRICS ───────────────────────────────────────────────────

hard = [(cid, ok, resp, tp, fp, fn, p, r, f1)
        for cid, ok, adv, resp, tp, fp, fn, p, r, f1 in results if not adv]
adv  = [(cid, ok, resp, tp, fp, fn, p, r, f1)
        for cid, ok, adv, resp, tp, fp, fn, p, r, f1 in results if adv]

hard_passed = sum(1 for _, ok, *_ in hard if ok)
adv_passed  = sum(1 for _, ok, *_ in adv  if ok)

print('SCORECARD · /dedup/corpus')
print(f'  Hard cases passing   : {hard_passed}/{len(hard)}')
print(f'  Advisory (LLM) cases : {adv_passed}/{len(adv)}')
print(f'    (advisory FN is informational; advisory FP is still a hard fail)')
print()

print('PER-CASE CLUSTER QUALITY (pairwise)')
print(f"  {'ID':<16} {'P':>5} {'R':>5} {'F1':>5}  {'TP':>3} {'FP':>3} {'FN':>3}  status")
print('  ' + '-' * 60)
for cid, ok, adv_flag, resp, tp, fp, fn, p, r, f1 in results:
    icon   = '✅' if ok else ('⚠️ ' if adv_flag else '❌')
    n_cl   = len(resp.get('clusters', [])) if resp else '?'
    n_sing = sum(1 for cl in resp.get('clusters', []) if len(cl) == 1) if resp else '?'
    print(f'  {cid:<16} {p:>5.2f} {r:>5.2f} {f1:>5.2f}  {tp:>3} {fp:>3} {fn:>3}'
          f'  {icon}  clusters={n_cl}  singletons={n_sing}')

print()
# Aggregate across hard cases with expected merges (skip empty-corpus case)
scored_hard = [(p, r, f1) for cid, ok, resp, tp, fp, fn, p, r, f1 in hard
               if resp and resp.get('clusters')]
if scored_hard:
    mp = sum(p for p,_,_ in scored_hard) / len(scored_hard)
    mr = sum(r for _,r,_ in scored_hard) / len(scored_hard)
    mf = sum(f for _,_,f in scored_hard) / len(scored_hard)
    print(f'  Mean (hard, non-empty)  P={mp:.2f}  R={mr:.2f}  F1={mf:.2f}')

In [ ]:
# ── TEARDOWN: delete test collections ─────────────────────────────────────
for _col in (COLLECTION, EMPTY_COLLECTION):
    try:
        _wv(f'schema/{_col}', 'DELETE')
        log.info('deleted collection  %s', _col)
    except requests.HTTPError as exc:
        log.warning('could not delete %s: %s', _col, exc)
print('teardown complete')